# 01 - Exploratory data analysis

Coverage, data quality and price levels of the analysis base (`marts.int_analysis_prices`: main period 2018-01-01..2025-10-31, valid rows, suspect-low excluded). All numbers come from saved queries in `analysis/queries/`.

In [1]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from mandipulse import viz
from mandipulse.viz import COMMODITIES, COMMODITY_COLORS, INK, INK_2, MUTED, load
pd.set_option("display.width", 160)
viz.style()

In [2]:
# States use categorical slots 4-7 of the reference palette in fixed order (validated separately).
STATE_COLORS = {"Gujarat": "#eda100", "Madhya Pradesh": "#e87ba4",
                "Maharashtra": "#008300", "Uttar Pradesh": "#4a3aa7"}

## Coverage: markets reporting per month

In [3]:
cov = load("phase3", "eda_01_markets_reporting_by_month")
cov["month"] = pd.to_datetime(cov["month"])
fig, ax = viz.figure("Markets reporting each month, by state",
                     "Distinct markets with >= 1 valid price (any crop) per month")
for state, g in cov.groupby("state"):
    ax.plot(g["month"], g["n_markets_reporting"], color=STATE_COLORS[state], label=state)
viz.label_line_ends(ax, cov, "month", "n_markets_reporting", "state")
ax.set_ylabel("markets reporting")
ax.set_ylim(0)
viz.legend(ax, ncols=4)
viz.save(fig, "eda_markets_reporting_by_month",
         note="Madhya Pradesh dips in Jan-Sep 2024 (known reporting gap); 2025 thins before the Nov-2025 source change.")
cov.pivot(index="month", columns="state", values="n_markets_reporting").tail(12)

state,Gujarat,Madhya Pradesh,Maharashtra,Uttar Pradesh
month,,,,
2024-11-01,37,62,98,197
2024-12-01,38,64,97,200
2025-01-01,37,66,100,198
2025-02-01,37,62,94,203
2025-03-01,37,62,96,202
2025-04-01,36,76,100,203
2025-05-01,35,81,100,206
2025-06-01,34,79,106,210
2025-07-01,34,78,102,210


In [4]:
load("phase3", "eda_02_coverage_by_state_crop")

,state,commodity,n_markets,n_market_days,median_report_days_per_market_year
0,Gujarat,Onion,45,37051.0,186.0
1,Gujarat,Potato,39,34058.0,205.0
2,Gujarat,Tomato,51,46457.0,214.0
3,Madhya Pradesh,Onion,135,73040.0,113.0
4,Madhya Pradesh,Potato,107,58296.0,120.0
5,Madhya Pradesh,Tomato,95,60857.0,149.0
6,Maharashtra,Onion,128,104030.0,131.0
7,Maharashtra,Potato,56,47865.0,153.0
8,Maharashtra,Tomato,70,79102.0,214.0
9,Uttar Pradesh,Onion,230,395417.0,273.0


## Data quality: share of rows flagged invalid, by year

In [5]:
flags = load("phase2", "02_flag_rates_by_year").dropna(subset=["year"])
flags["year"] = flags["year"].astype(int)
fig, ax = viz.figure("Rows flagged invalid, by year",
                     "% of staging rows failing >= 1 validity flag (order, outlier, unit); suspect-low is not counted")
ax.bar(flags["year"], flags["pct_invalid"], color=viz.SEQUENTIAL[4], width=0.6)
for x, y in zip(flags["year"], flags["pct_invalid"], strict=True):
    ax.annotate(f"{y:.2f}%", (x, y), xytext=(0, 3), textcoords="offset points", ha="center",
                fontsize=8.5, color=INK_2)
ax.set_ylabel("% of rows invalid")
ax.set_xticks(flags["year"])
viz.save(fig, "eda_invalid_rate_by_year")
flags[["year", "n_rows", "n_order", "n_outlier", "n_unit_suspect", "n_invalid", "n_suspect_low", "pct_invalid"]]

,year,n_rows,n_order,n_outlier,n_unit_suspect,n_invalid,n_suspect_low,pct_invalid
0,2018,220786,0,160,221,380,610,0.17
1,2019,228998,0,191,4,195,972,0.09
2,2020,210123,0,83,12,95,257,0.05
3,2021,227468,0,69,34,103,176,0.05
4,2022,241840,0,74,1,75,404,0.03
5,2023,239499,40,214,10,253,386,0.11
6,2024,225306,772,278,15,1022,445,0.45
7,2025,220590,0,546,11,555,462,0.25
8,2026,12332,0,13,6,19,6,0.15


## Price levels: monthly median modal price

In [6]:
px = load("phase3", "eda_03_median_price_monthly")
px["month"] = pd.to_datetime(px["month"])
fig, ax = viz.figure("Monthly median wholesale price by crop",
                     "Median of daily modal prices across all markets, Rs per quintal")
for c in COMMODITIES:
    g = px[px["commodity"] == c]
    ax.plot(g["month"], g["median_modal_rs_qtl"], color=COMMODITY_COLORS[c], label=c)
viz.label_line_ends(ax, px, "month", "median_modal_rs_qtl", "commodity")
ax.set_ylabel("Rs / quintal")
ax.set_ylim(0)
viz.legend(ax)
viz.save(fig, "eda_median_price_monthly",
         note="Tomato spikes (e.g. Jul 2023) and onion spikes (e.g. late 2019, 2020) are visible; prices are not inflation-adjusted.")
px.groupby("commodity")["median_modal_rs_qtl"].describe()

,count,mean,std,min,25%,50%,75%,max
commodity,,,,,,,,
Onion,94.0,1797.106383,887.180728,750.0,1188.75,1480.0,2115.0,5221.0
Potato,94.0,1204.787234,496.518478,500.0,871.25,1092.5,1332.5,3000.0
Tomato,94.0,1681.159574,1024.040535,600.0,962.50,1400.0,2100.0,7000.0
